# Nhận diện tấn công thông qua mô hình Logistic Regression

Ở notebook trước ([training-model-lr.ipynb](notebooks/training-model-lr.ipynb)), việc so sánh mô hình sạch và mô hình bị tấn công dựa trên thông tin đã biết trước: ta biết chính xác tập nào là `"0"` (sạch) và tập nào là `"45"` (bị đầu độc 45%), nên có thể so sánh trực tiếp accuracy và loss giữa hai kịch bản.

Trong thực tế, một hệ thống phòng thủ không biết trước liệu dữ liệu huấn luyện có bị đầu độc hay không, và đầu độc ở mức bao nhiêu phần trăm. Notebook này đi tìm các dấu hiệu gián tiếp (indirect signal) — những đại lượng có thể quan sát được mà không cần biết nhãn thật đã bị đảo ở đâu — để "nhận biết" (awareness) rằng mô hình đang được huấn luyện trên dữ liệu khả nghi. Đây là bước tiền đề cho phần phòng thủ (`src/defend`) sau này: muốn phòng thủ, trước tiên phải phát hiện được có tấn công.

Bốn case dưới đây khảo sát 4 loại dấu hiệu khác nhau, phần lớn quan sát theo % tỉ lệ nhãn bị đảo (flip) từ 0% đến 45%, dùng các mô hình đã huấn luyện sẵn ở `src/models` (`lr_trained, dt_trained` — mỗi model được train riêng cho từng mức % flip).

In [ ]:
import sys, os 
from pathlib import Path

project_root = Path.cwd().parent 

os.chdir(project_root)
sys.path.append(str(project_root))

In [ ]:
# Python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Project
from src.preprocess import X_test, y_test, train_dfs
from src.utils.math import *

## Chuẩn bị

Chuẩn bị dữ liệu và tham số dùng chung cho toàn bộ notebook:

Lấy tập huấn luyện bị đầu độc nặng nhất trong các case đơn lẻ (`train_dfs["45"]`, tức 45% nhãn bị đảo) làm ví dụ minh họa cho Case 1.
Chuẩn hóa (`StandardScaler`) cả `X_train` và `X_test` để tránh vấn đề weight nổ đã gặp ở các notebook trước.
Khởi tạo `w = [0.0, ..., 0.0]`, dùng `C_val = 0.01` và `epochs = 1000` — cùng cấu hình với notebook `training-model-lr.ipynb` để đảm bảo có thể so sánh chéo kết quả giữa hai notebook.

In [ ]:
# w weight
w = np.zeros(X_test.shape[1])

# Training data
training_data = train_dfs["45"]
X_train, y_train = training_data["X"], training_data["attack"]

# Scalling data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Const
C_val = 0.01
epochs = 1000

## 1. Trường hợp 1 - Khoảng cách giữa `train_loss` và `val_loss`

Ý tưởng: ở notebook trước, ta đã quan sát rằng mô hình huấn luyện trên dữ liệu sạch có train loss và test (val) loss bám khá sát nhau, trong khi mô hình huấn luyện trên dữ liệu bị đầu độc có khoảng cách (gap) lớn hơn. Case này đo trực tiếp đại lượng `gap = val_loss - train_loss` qua từng epoch, xem nó biến đổi thế nào khi huấn luyện trên tập bị đầu độc 45%.

### 1.1. Chuẩn bị

Khởi tạo `w_c1`, `b_c1 = 0`, huấn luyện gradient descent trên `X_train`/`y_train` (tập "45"), nhưng ở mỗi epoch tính thêm:

- `train_loss`: loss trên chính tập train (bị đầu độc).
- `val_loss`: loss trên tập test sạch.
- `gap = val_loss - train_loss`, lưu lại vào danh sách `gap` để vẽ đồ thị.

In [ ]:
# Two weights in case 1
w_c1 = w.copy()
b_c1 = 0

gap = []

for epoch in range(epochs + 1):
    train_loss = loss(y_train, sigmoid(X_train @ w_c1 + b_c1))
    val_loss = loss(y_test, sigmoid(X_test @ w_c1 + b_c1))

    gap.append(val_loss - train_loss)

    dw, db = gradient(X_train, y_train, sigmoid(X_train @ w_c1 + b_c1))

    w_c1 -= C_val * dw
    b_c1 -= C_val * db

### 1.2. Vẽ biểu đồ

**Kết quả quan sát được:** `gap` **tăng đơn điệu** theo epoch — từ gần 0 lúc mới khởi tạo, tăng dần lên đến khoảng **0.46** ở epoch 1000, không có dấu hiệu chững lại hay giảm xuống.
 
Đây là khác biệt quan trọng so với mô hình sạch: nếu train trên dữ liệu sạch, gap giữa train loss và test loss thường tăng chậm rồi ổn định ở mức thấp (như quan sát ở đồ thị "Loss Curve - Cleaning" trước đó). Còn ở đây, gap **tăng liên tục không có dấu hiệu bão hòa** — càng huấn luyện lâu, mô hình càng "khớp" tốt hơn với nhãn bị đảo trên tập train, nhưng càng lệch xa hơn so với phân bố thật (tập test sạch).
 
→ **Dấu hiệu awareness #1:** nếu quan sát thấy gap giữa train loss và validation loss tăng liên tục, không hội tụ, qua nhiều epoch — đó là tín hiệu cảnh báo dữ liệu huấn luyện có thể đã bị nhiễm/đầu độc, ngay cả khi không biết trước nhãn nào bị sai.

In [ ]:
plt.plot(gap)
plt.xlabel("Epochs")
plt.ylabel("Gap")
plt.title("Case 1")
plt.grid(True, alpha=0.3)
plt.show()

## 2. Trường hợp 2 - Kiểm tra trọng số `norm`

**Ý tưởng:** khi mô hình bị buộc phải "giải thích" cho các nhãn mâu thuẫn nhau (một phần nhãn đúng, một phần bị đảo), nó thường phải đẩy trọng số lên cao hơn để cố gắng phân tách các điểm dữ liệu — vốn dĩ đang lẫn lộn tín hiệu — thành hai lớp. Case này đo độ lớn `||w||` (chuẩn Euclid của vector trọng số) của các mô hình `lr_trained` đã huấn luyện sẵn ở nhiều mức % flip khác nhau.

### 2.1. Chuẩn bị

Lấy toàn bộ mô hình Logistic Regression đã huấn luyện sẵn trong `src.models.lr_trained` (dict key là % flip dạng string), sắp xếp theo thứ tự % flip tăng dần, rồi tính `np.linalg.norm(model.coef_)` cho từng mô hình.

In [ ]:
from src.models import lr_trained

w_nomrs = []
flip_percent = []

sorted_lr_trained = dict(
    sorted(lr_trained.items(), key=lambda x: int(x[0]))
)

for percent, model in sorted_lr_trained.items():
    w_nomrs.append(np.linalg.norm(model.coef_))
    flip_percent.append(int(percent))

### 2.2. Vẽ biểu đồ

**Kết quả quan sát được:** `||w||` tăng gần như tuyến tính từ **≈ 4.4** (0% flip) lên đến đỉnh **≈ 12.2** (40% flip) — gấp gần 3 lần — rồi giảm nhẹ xuống **≈ 11.8** ở 45%.
 
Xu hướng tăng mạnh này khớp với giả thuyết ban đầu: càng nhiều nhãn bị đảo, mô hình càng phải "cố" tăng độ dốc quyết định (trọng số lớn hơn) để cố gắng phân tách dữ liệu, dù bản chất dữ liệu đã bị nhiễu và không còn khả năng phân tách rạch ròi như ban đầu. Phần giảm nhẹ ở 40–45% có thể do ở mức đầu độc quá cao, ranh giới quyết định gần như ngẫu nhiên, khiến `lbfgs` solver (ghi nhận `ConvergenceWarning` trong log chạy) không còn cố "kéo căng" trọng số theo hướng rõ ràng nào nữa.
 
→ **Dấu hiệu awareness #2:** `||w||` bất thường lớn so với baseline (mô hình huấn luyện trên dữ liệu tin cậy) là tín hiệu cho thấy mô hình có thể đang bị buộc khớp với nhãn nhiễu — có thể dùng làm một dạng "anomaly threshold" để cảnh báo.

In [ ]:
plt.plot(flip_percent, w_nomrs, marker="o")
plt.xlabel("% flip")
plt.ylabel("||w||")
plt.title("Case 2")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Trường hợp 3 - So sánh với `một model khác`

**Ý tưởng:** nếu huấn luyện hai loại mô hình khác nhau (ở đây là Logistic Regression — mô hình tuyến tính, và Decision Tree — mô hình phi tuyến) trên cùng một tập dữ liệu, và dữ liệu đó sạch, hai mô hình thường sẽ đồng thuận (dự đoán giống nhau) trên phần lớn các mẫu, vì cả hai đều đang học cùng một quy luật thật. Nếu dữ liệu bị đầu độc, các mô hình có kiến trúc khác nhau sẽ phản ứng khác nhau với nhiễu, khiến tỉ lệ bất đồng (disagreement) giữa chúng tăng lên.

### 3.1. Chuẩn bị

Với mỗi mức % flip, lấy cặp mô hình `lr_model` (`lr_trained`) và `dt_model` (`dt_trained`) đã huấn luyện sẵn cùng mức đó, dự đoán trên cùng `X_test`, rồi tính `disagreement_rate = np.mean(y_lr_pred != y_dt_pred)` — tỉ lệ phần trăm mẫu mà hai mô hình đưa ra dự đoán khác nhau.

In [ ]:
from src.models import dt_trained

disagreement_rates = []

for percent in flip_percent:
    lr_model = lr_trained[str(percent)]
    dt_model = dt_trained[str(percent)]

    y_lr_pred = lr_model.predict(X_test)
    y_dt_pred = dt_model.predict(X_test)

    disagreement_rates.append(np.mean(y_lr_pred != y_dt_pred))

### 3.2. Vẽ biểu đồ

**Kết quả quan sát được:** disagreement rate tăng đều từ **≈ 8%** (0% flip) lên đến **≈ 41%** (45% flip) — tăng hơn 5 lần.
 
Ở mức 0% flip, hai mô hình khác kiến trúc vẫn đồng thuận đến 92% số mẫu — hợp lý vì cả hai đều học đúng ranh giới quyết định thật. Nhưng khi tỉ lệ flip tăng, LR (mô hình tuyến tính, nhạy với nhiễu nhãn vì tối ưu toàn cục theo trung bình gradient) và DT (mô hình phi tuyến, có xu hướng "chia nhỏ" không gian để khớp cả điểm nhiễu cục bộ) phản ứng khác nhau trước cùng một tập nhãn bị nhiễu, khiến chúng ngày càng bất đồng.
 
→ **Dấu hiệu awareness #3:** huấn luyện song song 2 mô hình có "thiên kiến quy nạp" (inductive bias) khác nhau trên cùng một dữ liệu, rồi đo tỉ lệ bất đồng, là một kỹ thuật phát hiện dữ liệu khả nghi phổ biến trong thực tế (không cần biết nhãn thật) — disagreement rate cao bất thường so với baseline là dấu hiệu cảnh báo.

In [ ]:
plt.plot(flip_percent, disagreement_rates, marker="o")
plt.xlabel("% flip")
plt.ylabel("disagreement rate")
plt.title("Case 3")
plt.show()

## 4. Trường hợp 4 - Kiểm tra `accurancy` của model

**Ý tưởng:** đây là phép đo trực tiếp và trực quan nhất — khác với 3 case trên (vốn không cần biết nhãn thật của quá trình tấn công), case này giả định ta *có* một tập test đáng tin cậy để đo accuracy thật của từng mô hình `lr_trained` qua các mức % flip, dùng làm đường cong tham chiếu ("ground truth") để đối chiếu lại với các dấu hiệu gián tiếp ở Case 1–3.

### 4.1. Chuẩn bị

Với mỗi mức % flip, lấy `model = lr_trained[str(percent)]`, dự đoán trên `X_test`, tính `accuracy_score(y_test, model.predict(X_test))`, lưu vào `acc_history`.
 
*(Phần code có sẵn nhưng đang bị comment — `extreme_ratio`: tỉ lệ dự đoán "quá tự tin" — xác suất `< 0.05` hoặc `> 0.95`. Đây là một dấu hiệu awareness bổ sung tiềm năng: mô hình bị ép khớp dữ liệu nhiễu thường có xu hướng đưa ra dự đoán cực đoan hơn để "cố phân tách" các điểm mâu thuẫn. Có thể bật lại đoạn này nếu muốn khảo sát thêm.)*

In [ ]:
acc_history = []
extreme_ratio_history = []

for percent in flip_percent:
    model = lr_trained[str(percent)]
    probs = model.predict_proba(X_test)[:, 1]

    extreme_ratio = np.mean((probs < 0.05) | (probs > 0.95))
    extreme_ratio_history.append(extreme_ratio)

    acc = accuracy_score(y_test, model.predict(X_test))
    acc_history.append(acc)

### 4.2. Vẽ biểu đồ

**Kết quả quan sát được:** accuracy giảm gần như tuyến tính từ **≈ 96%** (0% flip, khớp với kết quả 96.44% ở notebook `01_training_model_lr.ipynb`) xuống còn **≈ 58%** ở 45% flip — gần chạm mức đoán ngẫu nhiên với bài toán nhị phân.
 
Đường cong này xác nhận rằng cả 3 dấu hiệu gián tiếp ở Case 1–3 (gap loss tăng, `||w||` tăng, disagreement rate tăng) đều **tương quan cùng chiều** với mức độ suy giảm accuracy thật — nghĩa là các dấu hiệu đó là những "proxy" hợp lý để nhận biết tấn công, ngay cả trong tình huống không có tập test đáng tin cậy để đo accuracy trực tiếp như ở đây.

In [ ]:
plt.plot(flip_percent, acc_history, marker="o", label="Accuracy scores")
plt.plot(flip_percent, extreme_ratio_history, marker="o", label="Extreme ratioes")

plt.xlabel("% flip")
plt.ylabel("")
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.05)

plt.legend()
plt.show()

## In conclusion

ốn thí nghiệm trong notebook này cho thấy tấn công label flipping để lại **nhiều dấu vết quan sát được** trên hành vi của mô hình, không chỉ thể hiện qua con số accuracy cuối cùng:
 
| Case | Dấu hiệu | Xu hướng khi % flip tăng (0% → 45%) | Cần biết nhãn thật? |
|---|---|---|---|
| 1 | Gap giữa train loss và val loss | Tăng liên tục, không hội tụ (→ ~0.46) | Cần tập val sạch |
| 2 | `\|\|w\|\|` (chuẩn trọng số) | Tăng mạnh (~4.4 → ~12.2, đỉnh ở 40%) | Không cần |
| 3 | Disagreement rate giữa LR và DT | Tăng mạnh (~8% → ~41%) | Không cần |
| 4 | Accuracy thật trên test sạch | Giảm mạnh (~96% → ~58%) | Cần tập test có nhãn đúng |
 
Điểm quan trọng nhất: **Case 2 và Case 3 không cần biết nhãn thật hay tập test sạch** — đây chính là hai dấu hiệu khả thi nhất để triển khai một cơ chế "awareness" thực tế, phát hiện dữ liệu khả nghi *trước khi* mô hình được đưa vào sử dụng, chỉ dựa trên hành vi nội tại của mô hình (độ lớn trọng số) hoặc sự đồng thuận giữa nhiều mô hình khác kiến trúc — mà không đòi hỏi một tập dữ liệu "ground truth" tốn kém để kiểm chứng.
 
Đây là nền tảng hợp lý cho bước tiếp theo trong `src/defend`: xây dựng một quy tắc cảnh báo tự động (ví dụ: ngưỡng `||w||` hoặc ngưỡng disagreement rate vượt quá baseline bao nhiêu độ lệch chuẩn thì gắn cờ "khả nghi"), thay vì phải chờ đo được accuracy sụt giảm mới biết dữ liệu đã bị tấn công.